# Lottery Ticket Hypothesis — Finding Sparse, Trainable Neural Networks (2018)

_Papers · Efficiency & Compression_

**A dense net hides a tiny subnetwork that, reset to its original starting weights, trains just as well alone.**

---

This notebook is a paced, step-by-step walkthrough of the **Lottery Ticket Hypothesis — Finding Sparse, Trainable Neural Networks (2018)** lesson. Run each cell, read the short note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We reproduce the lottery-ticket experiment in four steps: (1) a tiny worked prune-and-reset example, (2) a hard teacher-labelled task and the student net plus its training helper, (3) Iterative Magnitude Pruning to find a sparse mask, and (4) the headline contrast — the same mask reset to the original init versus a fresh random init.

### Step 1 — A worked prune-and-reset by hand

The lottery-ticket recipe in miniature. Starting from an original init `theta0` and the weights `thetaj` after some training, we rank weights by their *trained* magnitude, keep the largest half, and build a mask. The crucial twist: the survivors are reset to their **original** `theta0` values, not their trained values — `thetaj` is used only to choose *which* weights to keep.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Sanity-check the lesson's worked example: prune 50%, reset survivors to theta0.
theta0 = torch.tensor([0.30, -0.05, 0.80, -0.60])   # original init
thetaj = torch.tensor([0.20,  0.04, 0.90, -0.50])   # after j training steps

order = torch.argsort(thetaj.abs())                  # ascending by |trained value|
mask = torch.zeros(4)
mask[order[2:]] = 1.0                                 # keep the 2 largest -> [0, 0, 1, 1]
ticket = mask * theta0                               # reset survivors to theta0
print("worked example: mask =", mask.tolist(), " winning ticket =", ticket.tolist())
# worked example: mask = [0.0, 0.0, 1.0, 1.0]  winning ticket = [0.0, -0.0, 0.8, -0.6]

### Step 2 — A hard task, the student net, and a masked-training helper

We generate labels from a *fixed* random nonlinear teacher MLP — a task with real structure that a sparse net must work to fit. `make_net` builds the student; `linears` pulls out its weight-bearing layers; `load_init` copies a saved init into a net. The key helper is `train`: after every optimizer step it multiplies each layer's weights by its mask, forcing pruned weights to stay exactly zero throughout training.

In [ ]:
# A hard task: labels from a FIXED random nonlinear teacher MLP (D -> 64 -> K).
g = torch.Generator().manual_seed(2)
N, D, K = 2400, 40, 6
X = torch.randn(N, D, generator=g)
T1 = torch.randn(D, 64, generator=g) / (D ** 0.5)
T2 = torch.randn(64, K, generator=g) / (64 ** 0.5)
with torch.no_grad():
    y = (torch.relu(X @ T1) @ T2).argmax(1)          # the teacher's labels
Xtr, ytr, Xte, yte = X[:1600], y[:1600], X[1600:], y[1600:]

# The student net + helpers.
H = 300
def make_net():
    return nn.Sequential(nn.Linear(D, H), nn.ReLU(),
                         nn.Linear(H, H), nn.ReLU(),
                         nn.Linear(H, K))

def linears(net):
    return [m for m in net if isinstance(m, nn.Linear)]

def load_init(net, t0):
    with torch.no_grad():
        for p, p0 in zip(net.parameters(), t0):
            p.copy_(p0)

def train(net, masks, steps=300, lr=0.1):
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lossf = nn.CrossEntropyLoss()
    net.train()
    for t in range(steps):
        opt.zero_grad()
        lossf(net(Xtr), ytr).backward()
        opt.step()
        with torch.no_grad():                         # KEEP pruned weights at exactly 0
            for lyr, mk in zip(linears(net), masks):
                lyr.weight.mul_(mk)
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item()

### Step 3 — Save the original init, train dense, and run Iterative Magnitude Pruning

First we save the original init `theta0` (the value tickets reset to) and train a fully dense net as the reference. Then `imp` runs Iterative Magnitude Pruning: each round resets to `theta0`, trains, and drops a fraction of the smallest surviving weights — retraining between cuts so each magnitude ranking reflects a re-optimized net. After 13 rounds keeping 70% each, about `0.7^13 ≈ 1%` of weights remain.

In [ ]:
# Save the ORIGINAL init theta0 (the value the ticket resets to).
torch.manual_seed(7)
base = make_net()
theta0 = [p.detach().clone() for p in base.parameters()]
lins = linears(base)

dense = make_net()
load_init(dense, theta0)
dense_acc = train(dense, [torch.ones_like(l.weight) for l in linears(dense)])

# Iterative Magnitude Pruning: n rounds, keep 70% of survivors each round.
def imp(theta0, rounds=13, keep=0.7):
    masks = [torch.ones_like(l.weight) for l in lins]
    for r in range(rounds):
        net = make_net()
        load_init(net, theta0)
        with torch.no_grad():
            for lyr, mk in zip(linears(net), masks):
                lyr.weight.mul_(mk)
        train(net, masks)                              # train, THEN prune by magnitude
        for lyr, mk in zip(linears(net), masks):
            w = lyr.weight.detach().abs()
            alive = w[mk.bool()]
            k = int((1 - keep) * alive.numel())
            if k > 0:
                thr = torch.kthvalue(alive, k).values
                mk.copy_((w > thr).float() * mk)       # drop smallest survivors
    return masks

masks = imp(theta0, rounds=13, keep=0.7)               # 0.7^13 ~= 1% remaining
Pm = sum(m.sum().item() for m in masks) / sum(m.numel() for m in masks)

### Step 4 — The headline contrast: same mask, two different starting weights

This is the experiment that defines the paper. Holding the mask fixed, we train it two ways: `eval_ticket` resets the survivors to their original `theta0` (the winning ticket), while `eval_random` gives the *same wiring* a fresh random init. Everything else — structure, optimizer, data — is identical, so any gap isolates the initialization. The reset-to-init ticket trains close to dense; the random reinit lags.

In [ ]:
# The headline contrast: SAME mask, two different starting weights.
def eval_ticket():                                      # reset survivors to theta0
    net = make_net()
    load_init(net, theta0)
    with torch.no_grad():
        for lyr, mk in zip(linears(net), masks):
            lyr.weight.mul_(mk)
    return train(net, masks)

def eval_random(seed):                                  # SAME mask, FRESH random init
    torch.manual_seed(seed)
    net = make_net()
    with torch.no_grad():
        for lyr, mk in zip(linears(net), masks):
            lyr.weight.mul_(mk)
    return train(net, masks)

ticket_acc = eval_ticket()
random_accs = [eval_random(100 + i) for i in range(5)]

print(f"\nDENSE test acc          : {dense_acc:.4f}")
print(f"Sparsity Pm (remaining) : {Pm*100:.2f}%   (pruned {(1-Pm)*100:.2f}%)")
print(f"WINNING TICKET (reset)  : {ticket_acc:.4f}")
print(f"RANDOM REINIT (same mask): {[round(a,4) for a in random_accs]}")
# At ~1% remaining, the reset-to-init ticket trains close to dense; the random-reinit
# (same wiring, fresh init) lags. Exact numbers vary by hardware/seed --
# this is our small run, not the paper's reported result.

## Visualize the data & results

_At 99% sparsity, does the winning ticket (survivors reset to their ORIGINAL init) out-train the SAME sparse wiring given a fresh RANDOM init?_

### Step 1 — Rebuild the task, net, and a loss-recording training helper

This visualization cell stands alone, so it re-creates the teacher task, the student net, and the helpers. The `train` helper gains a `rec` flag: when set, it records the training loss every 10 steps so we can later compare the *learning curves*, not just final accuracy. We also retrain the dense baseline here.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Reproduce the headline contrast: at high sparsity, a winning ticket (survivors
# RESET to the original init) trains; the SAME mask with a fresh random init stalls.
g = torch.Generator().manual_seed(2)
N, D, K = 2400, 40, 6
X = torch.randn(N, D, generator=g)
T1 = torch.randn(D, 64, generator=g) / (D ** 0.5)      # fixed random nonlinear teacher
T2 = torch.randn(64, K, generator=g) / (64 ** 0.5)
with torch.no_grad():
    y = (torch.relu(X @ T1) @ T2).argmax(1)
Xtr, ytr, Xte, yte = X[:1600], y[:1600], X[1600:], y[1600:]

H = 300
def make_net():
    return nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, H), nn.ReLU(), nn.Linear(H, K))

def lins(net):
    return [m for m in net if isinstance(m, nn.Linear)]

def load(net, t0):
    with torch.no_grad():
        for p, p0 in zip(net.parameters(), t0):
            p.copy_(p0)

def train(net, masks, steps=300, lr=0.1, rec=False):
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    lossf = nn.CrossEntropyLoss()
    net.train()
    curve = []
    for t in range(steps):
        opt.zero_grad()
        loss = lossf(net(Xtr), ytr)
        loss.backward()
        opt.step()
        with torch.no_grad():
            for l, m in zip(lins(net), masks):
                l.weight.mul_(m)                          # keep pruned weights at 0
        if rec and t % 10 == 0:
            curve.append([t, round(loss.item(), 4)])
    net.eval()
    with torch.no_grad():
        acc = (net(Xte).argmax(1) == yte).float().mean().item()
    return (acc, curve) if rec else acc

torch.manual_seed(7)
base = make_net()
theta0 = [p.detach().clone() for p in base.parameters()]
L = lins(base)
dn = make_net()
load(dn, theta0)
dense = train(dn, [torch.ones_like(l.weight) for l in lins(dn)])

### Step 2 — Find the mask, then plot the two learning curves

We run IMP to ~1% remaining, then train the mask two ways with loss recording on. `ticket` resets survivors to `theta0`; `randinit` gives the same mask a fresh random init. Printing both learning curves side by side shows the ticket's loss falling steadily while the random reinit stalls near its starting loss — the contrast at 99% sparsity, made visible.

In [ ]:
def imp(t0, rounds=13, keep=0.7):
    masks = [torch.ones_like(l.weight) for l in L]
    for r in range(rounds):
        net = make_net()
        load(net, t0)
        with torch.no_grad():
            for l, m in zip(lins(net), masks):
                l.weight.mul_(m)
        train(net, masks)
        for l, m in zip(lins(net), masks):
            w = l.weight.detach().abs()
            a = w[m.bool()]
            k = int((1 - keep) * a.numel())
            if k > 0:
                m.copy_((w > torch.kthvalue(a, k).values).float() * m)
    return masks

masks = imp(theta0)                                     # ~1% remaining
Pm = sum(m.sum().item() for m in masks) / sum(m.numel() for m in masks)

def ticket():
    net = make_net()
    load(net, theta0)
    with torch.no_grad():
        for l, m in zip(lins(net), masks):
            l.weight.mul_(m)
    return train(net, masks, rec=True)

def randinit(seed, rec=False):
    torch.manual_seed(seed)
    net = make_net()
    with torch.no_grad():
        for l, m in zip(lins(net), masks):
            l.weight.mul_(m)
    return train(net, masks, rec=rec)

wt_acc, wt_curve = ticket()
rr_acc, rr_curve = randinit(100, rec=True)
rr_accs = [randinit(100 + i) for i in range(5)]
print("DENSE        :", round(dense, 4))
print("Pm remaining :", round(Pm * 100, 2), "%")
print("TICKET acc   :", round(wt_acc, 4))
print("RANDOM accs  :", [round(a, 4) for a in rr_accs], "mean", round(float(np.mean(rr_accs)), 4))
print("ticket curve :", wt_curve)
print("random curve :", rr_curve)
# Ticket loss 1.08 -> 0.31 (acc ~0.78); random stalls near 1.0, ends ~0.89 (acc ~0.71).
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline ablation. You have a winning ticket at high sparsity (say 1&percnt; of weights
            remaining) that trains nearly as well as the dense net. You keep its exact mask but replace the
            surviving weights with a fresh random initialization, then retrain. What happens to the
            training curve and final accuracy, and what does the contrast prove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Hold everything fixed except the starting weights: same mask, same depth/width, same optimizer, same data. Only swap reset-to-$\theta_0$ for a fresh random init. — _An honest ablation changes exactly one thing &mdash; the starting values &mdash; so any difference is attributable to the initialization, not the structure._
- Retrain and watch: the winning ticket's loss falls steadily to a low value; the random-reinit net (same wiring) stalls near its starting loss far longer and ends higher. — _If structure alone explained the ticket, the random-reinit net would train just as well. It does not, so the original init $\theta_0$ is doing real work._
- Conclude that the specific starting weights $\theta_0$, paired with that mask, are what win &mdash; not the sparse wiring by itself. — _This is exactly the paper's claim (&sect;2): "structure alone cannot explain a winning ticket's success."_

**Answer:** The random-reinit subnetwork (same mask, fresh init) trains worse: its loss stalls
                 longer and it reaches lower final accuracy than the winning ticket. Since the two nets
                 are identical except for the starting weights, this isolates the initialization &mdash;
                 not the structure &mdash; as the reason the ticket trains. The CODEVIZ panel shows exactly this
                 contrast at 99&percnt; sparsity.

</details>

**Problem 2.** Work the prune-and-reset by hand. A 4-weight layer starts at
            $\theta_0 = [0.10,\,-0.70,\,0.05,\,0.40]$ and after training is
            $\theta_j = [0.30,\,-0.65,\,0.02,\,0.55]$. Magnitude-prune 50&percnt;, then write the winning
            ticket. Which surviving values go in?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Take magnitudes of the trained weights: $|\theta_j| = [0.30,\,0.65,\,0.02,\,0.55]$. — _Magnitude pruning ranks by absolute trained value; small means least important._
- Delete the two smallest: $0.02$ (weight 3) and $0.30$ (weight 1). Mask $m = [0,\,1,\,0,\,1]$. — _50&percnt; of 4 weights is 2; the survivors are weights 2 and 4, the two largest by magnitude._
- Reset survivors to $\theta_0$: ticket $= m \odot \theta_0 = [0,\,-0.70,\,0,\,0.40]$. — _The key step: survivors take their original init values $-0.70$ and $0.40$, NOT the trained $-0.65$ and $0.55$._

**Answer:** $m = [0,1,0,1]$ and the winning ticket is $m \odot \theta_0 = [0,\,-0.70,\,0,\,0.40]$.
                 The two survivors are reset to their original init values $-0.70$ and $0.40$ &mdash;
                 the trained values $-0.65$ and $0.55$ were used only to pick the mask, then thrown
                 away.

</details>

**Problem 3.** Why does Iterative Magnitude Pruning (many small rounds, $p^{1/n}\%$ each) find sparser winning
            tickets than one-shot pruning (one big $p\%$ cut)? Give the intuition and a 3-round example
            reaching ~50&percnt; sparsity.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that pruning the smallest weights of a just-trained net is a better estimate of "unimportant" than pruning a net that was never retrained after a huge cut. — _Each IMP round retrains the survivors before the next cut, so each magnitude ranking reflects a network that has re-optimized around its remaining weights._
- Pick a per-round keep rate. To keep ~50&percnt; over 3 rounds, keep $0.5^{1/3} \approx 0.794$ each round: $0.794^3 \approx 0.50$. — _Compounding small cuts reaches the target sparsity while letting the net recover between cuts._
- Contrast one-shot: a single 50&percnt; cut ranks weights once, on a net that never adapts to the loss of half its weights. — _The one-shot ranking is noisier, so it deletes some weights the iterative process would have kept, giving a weaker ticket._

**Answer:** Iterative pruning retrains between cuts, so each magnitude ranking is taken on a network that
                 has re-optimized around the weights it still has &mdash; a cleaner signal of which weights are
                 truly unimportant. Compounding gentle cuts (e.g. keep $0.794$ per round for 3 rounds
                 $\to 0.794^3 \approx 0.50$) reaches the same sparsity as one big cut but yields a stronger
                 ticket, which is why IMP reaches the paper's $\lt 10\text{-}20\%$ regime.

</details>